# Capítulo 5: Estimação de Parâmetros em Sistemas Biológicos
## Exemplos Práticos de Ajuste de Modelos, Otimização e Incerteza em Python

Este notebook reúne os exemplos práticos discutidos no Capítulo 5 da disciplina **Bioinformática para Biologia de Sistemas**. Aqui você encontrará implementações de:
1. Busca em grade unidimensional (Grid Search 1D)
2. Busca em grade bidimensional (Grid Search 2D) com list comprehension e visualização de contorno (Contour Plot)
3. Descida de Gradiente (Gradient Descent) com cálculo de gradiente numérico
4. Estimação local de parâmetros usando scipy com `curve_fit` e `least_squares` (Levenberg-Marquardt)
5. Otimização global de parâmetros usando `differential_evolution`
6. Análise de correlação e matriz de covariância
7. Cálculo de Intervalos de Confiança (ICs) assintóticos de 95% usando a distribuição t de Student e visualização gráfica
8. Validação e cálculo de métricas de qualidade de ajuste ($R^2$, RMSE, AIC e BIC)

--- 
## Exemplo 1: Busca em Grade Unidimensional (Grid Search 1D)

O Grid Search varre sistematicamente um intervalo pré-definido avaliando a função objetivo para encontrar o ponto com menor erro. Aqui minimizamos a função de custo $J(\theta) = (\theta - 3)^2 + 5$ no intervalo $\theta \in [0, 6]$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definir função objetivo a minimizar
def objective_function(theta):
    return (theta - 3)**2 + 5

# Definir a grade de valores para theta
theta_values = np.linspace(0, 6, 100)

# Avaliar a função em cada nó
J_values = [objective_function(t) for t in theta_values]

# Encontrar o índice do menor valor de custo
idx_min = np.argmin(J_values)
theta_best = theta_values[idx_min]
J_best = J_values[idx_min]

print(f"Parâmetro ótimo (theta): {theta_best:.3f}")
print(f"Custo mínimo J(theta): {J_best:.3f}")

# Visualizar a curva de custo
plt.figure(figsize=(8, 4.5))
plt.plot(theta_values, J_values, 'b-', linewidth=2, label='J(theta)')
plt.plot(theta_best, J_best, 'ro', markersize=8, label=f'Mínimo Estimado: theta={theta_best:.2f}')
plt.xlabel('Parâmetro theta')
plt.ylabel('Custo J(theta)')
plt.title('Busca em Grade 1D: Landscape de Custo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- 
## Exemplo 2: Busca em Grade Bidimensional (Grid Search 2D) e Contour Plot

Estendemos a busca em grade para duas dimensões para ajustar parâmetros $a$ e $b$ de um modelo exponencial de decaimento:
$$y(t) = a e^{-b t}$$
Abaixo avaliamos o erro quadrático (SSE) em uma grade regular de parâmetros e plotamos o mapa de contornos (Contour Plot) correspondente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Modelo de decaimento exponencial
def sse_model(params, t, y):
    a, b = params
    y_pred = a * np.exp(-b * t)
    return np.sum((y - y_pred)**2)

# Dados experimentais sintéticos
t_data = np.array([0, 1, 2, 3, 4, 5])
y_data = np.array([10.0, 6.1, 3.7, 2.2, 1.4, 0.8])

# Grade de parâmetros
a_range = np.linspace(5, 15, 50)
b_range = np.linspace(0.1, 1.0, 50)

# Avaliar SSE em toda a grade 2D
SSE_grid = np.array([[sse_model([a, b], t_data, y_data) for b in b_range] for a in a_range])

# Encontrar o índice do menor SSE
idx = np.unravel_index(SSE_grid.argmin(), SSE_grid.shape)
a_best = a_range[idx[0]]
b_best = b_range[idx[1]]
sse_min = SSE_grid[idx]

print(f"Parâmetros ótimos encontrados na busca em grade 2D:")
print(f"a = {a_best:.3f}, b = {b_best:.3f} (SSE mínimo: {sse_min:.4f})")

# Visualização da Landscape da Função Objetivo
A, B = np.meshgrid(a_range, b_range)

plt.figure(figsize=(9, 6.5))
contour = plt.contourf(A, B, SSE_grid.T, levels=30, cmap='viridis')
plt.colorbar(contour, label='Soma dos Quadrados dos Resíduos (SSE)')

# Adicionar linhas de contorno em branco e marcar o mínimo
plt.contour(A, B, SSE_grid.T, levels=15, colors='white', linewidths=0.5, alpha=0.4)
plt.plot(a_best, b_best, 'r*', markersize=15, label=f'Mínimo: a={a_best:.2f}, b={b_best:.2f}')

plt.xlabel('Parâmetro a')
plt.ylabel('Parâmetro b')
plt.title('Landscape da Função Objetivo: Contour Plot (Decaimento Exponencial)')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

--- 
## Exemplo 3: Algoritmo Manual de Gradient Descent (Descida do Gradiente)

O método da descida do gradiente avança iterativamente na direção oposta ao gradiente local da função objetivo. Usamos diferenças finitas centrais para estimar numericamente o vetor gradiente:
$$\nabla J = \left[\frac{\partial J}{\partial \theta_1}, \frac{\partial J}{\partial \theta_2}, \dots\right]^T$$

In [ ]:
import numpy as np

def numerical_gradient(f, theta, h=1e-5):
    """Calcula gradiente numericamente por diferenças finitas centrais"""
    grad = np.zeros_like(theta)
    for i in range(len(theta)):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[i] += h
        theta_minus[i] -= h
        grad[i] = (f(theta_plus) - f(theta_minus)) / (2 * h)
    return grad

def gradient_descent(f, theta0, alpha=0.01, max_iter=200, tol=1e-6):
    """Implementação básica do algoritmo Gradient Descent"""
    theta = np.array(theta0, dtype=float)
    history = [theta.copy()]
    
    for k in range(max_iter):
        grad = numerical_gradient(f, theta)
        
        # Atualização pelo negativo do gradiente
        theta_new = theta - alpha * grad
        
        # Verificar critério de parada (norma da mudança de parâmetros)
        if np.linalg.norm(theta_new - theta) < tol:
            print(f"Convergência atingida na iteração {k+1}.")
            break
            
        theta = theta_new
        history.append(theta.copy())
        
    return theta, np.array(history)

# Testar com uma função quadrática 2D simples: J(x, y) = (x-2)^2 + (y-3)^2 + 10
def f_test(theta):
    x, y = theta
    return (x - 2)**2 + (y - 3)**2 + 10

# Ponto inicial distante do mínimo
theta_init = [0.0, 0.0]

# Executar
theta_opt, hist = gradient_descent(f_test, theta_init, alpha=0.1)

print(f"Mínimo ótimo estimado: x = {theta_opt[0]:.4f}, y = {theta_opt[1]:.4f}")
print(f"Custo final obtido: {f_test(theta_opt):.6f}")

--- 
## Exemplo 4: Otimização Local de Curvas (Cinética de Michaelis-Menten com scipy)

Modelamos a velocidade de reação enzimática $v$ em função da concentração de substrato $S$ usando a equação de Michaelis-Menten:
$$v = \frac{V_{\max} S}{K_M + S}$$
Utilizamos `scipy.optimize.curve_fit` e `scipy.optimize.least_squares` (Levenberg-Marquardt) para estimar $V_{\max}$ e $K_M$ de dados ruidosos.

In [ ]:
from scipy.optimize import curve_fit, least_squares
import numpy as np
import matplotlib.pyplot as plt

# Definição do modelo matemático
def michaelis_menten(S, Vmax, Km):
    return Vmax * S / (Km + S)

# Dados experimentais reais simulados
S_data = np.array([0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
v_data = np.array([0.4, 0.7, 1.2, 1.8, 2.1, 2.3, 2.4])

# 1. Ajuste local simples usando curve_fit
p0_chute = [3.0, 5.0]
popt, pcov = curve_fit(michaelis_menten, S_data, v_data, p0=p0_chute)
Vmax_fit, Km_fit = popt

print("--- AJUSTE VIA curve_fit ---")
print(f"Vmax = {Vmax_fit:.4f}")
print(f"Km = {Km_fit:.4f}")

# 2. Ajuste usando least_squares com algoritmo de Levenberg-Marquardt
def residuals(params):
    Vmax, Km = params
    return v_data - michaelis_menten(S_data, Vmax, Km)

res_lm = least_squares(residuals, x0=p0_chute, method='lm')

print("\n--- AJUSTE VIA least_squares (Levenberg-Marquardt) ---")
print(f"Parâmetros estimados: Vmax={res_lm.x[0]:.4f}, Km={res_lm.x[1]:.4f}")
print(f"Custo mínimo final: {res_lm.cost:.6f}")
print(f"Número de avaliações da função: {res_lm.nfev}")

# Visualizar o ajuste
S_dense = np.linspace(0, 55, 200)
v_pred = michaelis_menten(S_dense, Vmax_fit, Km_fit)

plt.figure(figsize=(8, 5))
plt.scatter(S_data, v_data, color='red', s=50, zorder=5, label='Dados Experimentais')
plt.plot(S_dense, v_pred, 'b-', linewidth=2, label=f'Modelo Ajustado (Vmax={Vmax_fit:.2f}, Km={Km_fit:.2f})')
plt.xlabel('Concentração de Substrato [S]')
plt.ylabel('Velocidade da Reação [v]')
plt.title('Ajuste de Curva Cinética de Michaelis-Menten')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- 
## Exemplo 5: Otimização Global com Differential Evolution (DE)

Em problemas biológicos mais complexos, a presença de múltiplos mínimos locais impossibilita o sucesso de métodos locais tradicionais que dependem fortemente de um bom chute inicial. O algoritmo de evolução diferencial é ideal para fazer busca global.

In [ ]:
from scipy.optimize import differential_evolution
import numpy as np

# Função objetivo (SSE) do decaimento exponencial
def sse_global(params, t, y):
    a, b = params
    y_pred = a * np.exp(-b * t)
    return np.sum((y - y_pred)**2)

# Dados com ruído artificial
t_exp = np.linspace(0, 10, 15)
y_exp = 5.0 * np.exp(-0.4 * t_exp) + np.random.normal(0, 0.2, len(t_exp))

# Definir intervalos limites (bounds) de busca
bounds = [(0.1, 20.0),   # Limite para o parâmetro 'a'
          (0.01, 5.0)]   # Limite para o parâmetro 'b'

# Executar algoritmo evolucionário global
res_de = differential_evolution(sse_global, bounds, args=(t_exp, y_exp), seed=42)

print("--- AJUSTE GLOBAL VIA EVOLUÇÃO DIFERENCIAL ---")
print(f"Parâmetros estimados: a={res_de.x[0]:.4f}, b={res_de.x[1]:.4f}")
print(f"SSE mínimo obtido: {res_de.fun:.6f}")
print(f"Sucesso na otimização? {res_de.success}")

--- 
## Exemplo 6: Análise de Correlação e Identificabilidade Prática

A identificabilidade prática avalia se a qualidade dos dados experimentais é suficiente para separar os parâmetros. Analisamos a correlação a partir da matriz de covariância:
$$\text{Corr}_{ij} = \frac{\text{Cov}_{ij}}{\sigma_i \sigma_j}$$
Abaixo plotamos a matriz de correlação correspondente ao ajuste local de Michaelis-Menten executado no Exemplo 4.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Extrair desvios padrão marginais (erros padrão dos parâmetros)
std_errors = np.sqrt(np.diag(pcov))

# Normalizar matriz de covariância pcov para obter matriz de correlação
corr_matrix = pcov / np.outer(std_errors, std_errors)

# Exibir matriz de correlação
param_names = ['Vmax', 'Km']
print("Matriz de Correlação dos Parâmetros:")
print(corr_matrix)

# Visualizar com heatmap do Seaborn
plt.figure(figsize=(6, 4.5))
sns.heatmap(corr_matrix, annot=True, fmt='.3f',
            xticklabels=param_names, yticklabels=param_names,
            cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True)
plt.title('Matriz de Correlação (Parâmetros Michaelis-Menten)')
plt.show()

# Alerta de Identificabilidade Prática
if abs(corr_matrix[0, 1]) > 0.9:
    print(f"ALERTA: Alta correlação detectada (r = {corr_matrix[0, 1]:.3f}).")
    print("Os parâmetros podem estar acoplados, dificultando a identificabilidade prática.")
else:
    print("Os parâmetros possuem correlação moderada ou baixa, indicando boa identificabilidade prática.")

--- 
## Exemplo 7: Cálculo de Intervalos de Confiança Assintóticos (ICs 95%)

Utilizamos a matriz de covariância estimada e a distribuição t de Student para calcular intervalos de confiança de 95% para os parâmetros cinéticos do ajuste local de Michaelis-Menten.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

# Número de dados e parâmetros
n = len(S_data)
p = len(popt)
dof = max(0, n - p) # Graus de liberdade

# Definir nível de significância alpha (95% de confiança => alpha=0.05)
alpha = 0.05
t_value = stats.t.ppf(1 - alpha / 2, dof)

# Margens de erro dos intervalos de confiança (yerr)
ci_margins = t_value * std_errors

print("Intervalos de Confiança de 95% (Assintóticos):")
for i, name in enumerate(param_names):
    print(f"{name}: {popt[i]:.4f} +/- {ci_margins[i]:.4f} (Erro Relativo: {100*std_errors[i]/popt[i]:.2f}%)")

# Visualizar parâmetros estimados com barras de erro
plt.figure(figsize=(7, 4.5))
x_pos = np.arange(len(param_names))
plt.bar(x_pos, popt, yerr=ci_margins, alpha=0.7, capsize=10, color='steelblue', edgecolor='black')
plt.xticks(x_pos, param_names)
plt.ylabel('Valor Estimado do Parâmetro')
plt.title('Estimações de Parâmetros com Intervalos de Confiança (95%)')
plt.grid(axis='y', alpha=0.3)
plt.show()

--- 
## Exemplo 8: Métricas de Validação e Seleção de Modelos

Calculamos métricas estatísticas para avaliar a qualidade e a complexidade do ajuste: o Erro Quadrático Médio (RMSE), Coeficiente de Determinação ($R^2$), $R^2$ ajustado, além dos critérios de informação AIC e BIC para seleção de modelos.

In [ ]:
import numpy as np

def calculate_validation_metrics(y_true, y_pred, n_params):
    n = len(y_true)
    residuals = y_true - y_pred
    sse = np.sum(residuals**2)
    
    # 1. RMSE (Root Mean Squared Error)
    rmse = np.sqrt(sse / n)
    
    # 2. R2 (R-squared) e R2 ajustado
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - (sse / ss_tot)
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - n_params - 1)
    
    # 3. Critérios de Seleção AIC e BIC (fórmulas aproximadas baseadas em SSE)
    # Sob hipótese gaussiana, a log-verossimilhança é proporcional a n*log(sse/n)
    aic = n * np.log(sse / n) + 2 * n_params
    bic = n * np.log(sse / n) + n_params * np.log(n)
    
    return {
        'SSE': sse,
        'RMSE': rmse,
        'R2': r2,
        'R2_adjusted': r2_adj,
        'AIC': aic,
        'BIC': bic
    }

# Calcular previsões nos mesmos pontos experimentais do ajuste de Michaelis-Menten
v_pred_exp = michaelis_menten(S_data, Vmax_fit, Km_fit)

# Obter métricas
metrics = calculate_validation_metrics(v_data, v_pred_exp, n_params=2)

print("Métricas de Validação para Ajuste Michaelis-Menten:")
for k, v in metrics.items():
    print(f"  {k:12}: {v:.6f}")